In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
using System.IO;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

In [ ]:
// var myBatch = ExecutionQueues[1];
// myBatch

In [ ]:
//BoSSSshell.WorkflowMgm.Init("RisingBubble2D", myBatch);

In [ ]:
var db = OpenOrCreateDatabase(@"P:\hpccluster\RisingBubble2D");

Opening existing database 'P:\hpccluster\RisingBubble2D'.


In [ ]:
// var sessions = BoSSSshell.WorkflowMgm.Sessions;
var sessions = db.Sessions;
sessions

#0: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_ReInit100_testcase1	08/30/2024 09:22:51	44e79500...
#1: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_Picard_FastMarching_ReInit50_testcase1	08/30/2024 09:35:54	46f02668...
#2: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart2*	08/29/2024 10:31:41	d3fbafe6...
#3: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withPreEnforcerActive_restart1*	08/28/2024 15:08:37	d55fdec7...
#4: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1*	08/27/2024 14:57:24	d9c4a861...


In [ ]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [ ]:
var sess = sessions.Pick(1);
sess

RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_Picard_FastMarching_ReInit50_testcase1	08/30/2024 09:35:54	46f02668...

In [ ]:
evalSess.Add(sess);

In [ ]:
//sess.Delete(true);

In [ ]:
//sess.Timesteps.TakeLast(10).Export().WithSupersampling(3).Do()

C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\RisingBubble2D__RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart2__d3fbafe6-263c-4031-ab07-a8453141fd8e

In [ ]:
string studyName = "RB2D_fullDomain_20x40AMR0_k3_testcase1";
foreach (var s in sessions) {
    if (s.Name.Contains(studyName)) {
        evalSess.Add(s);
    }
}
evalSess

#0: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart2*	08/29/2024 10:31:41	d3fbafe6...
#1: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withPreEnforcerActive_restart1*	08/28/2024 15:08:37	d55fdec7...
#2: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1*	08/27/2024 14:57:24	d9c4a861...


# Evaluation - rising bubble benchmark quantities

In [ ]:
public static void PerformPostProcessing(ISessionInfo evalS) {

    // set up log 
    var db = databases.Pick(0);
    TextWriter Log = db.Controller.DBDriver.FsDriver.GetNewLog(RisingBubble2DBenchmarkQuantities.LogfileName, evalS.ID);
    string header = String.Format("{0}\t{1}\t{2}\t{3}\t{4}\t{5}\t{6}\t{7}", "#timestep", "time", "area", "center of mass - x", "center of mass - y", "circularity", "mean velocity - x", "mean velocity - y");
    Log.WriteLine(header);
    Log.Flush();

    // perform postprocessing for each time step
    foreach(var tStep in evalS.Timesteps) {

        // set up LsTrk and velocity fields   
        SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
        GridData grdData = (GridData)phi.GridDat; 
        LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
        levelSet.Acc(1.0, phi); 
        LevelSetTracker LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
        LsTrk.UpdateTracker(tStep.PhysicalTime);

        int D = grdData.SpatialDimension;
        XDGField[] VelocityFields = new XDGField[D];
        VelocityFields[0] = (XDGField)tStep.GetField("VelocityX");
        VelocityFields[1] = (XDGField)tStep.GetField("VelocityY");

        // compute benchmark quantities
        var R = RisingBubbleBenchmarkQuantitites.ComputeBenchmarkQuantities_RisingBubble2D(LsTrk, VelocityFields);

        // write line to log
        string line = String.Format("{0}\t{1}\t{2}\t{3}\t{4}\t{5}\t{6}\t{7}", tStep.TimeStepNumber.MajorNumber, tStep.PhysicalTime, 
                        R.area, R.centerX, R.centerY, R.circularity, R.MeanVelocityX, R.MeanVelocityY);
        Log.WriteLine(line);
        Log.Flush();
    }

    // Dispose
    Log.Close();
    Log.Dispose();

}

In [ ]:
List<Plot2Ddata> evalData = evalSess.ReadLogDataForXNSE(RisingBubble2DBenchmarkQuantities.LogfileName);

Element at 0: time vs area
Element at 1: time vs center of mass - x
Element at 2: time vs center of mass - y
Element at 3: time vs circularity
Element at 4: time vs mean velocity - x
Element at 5: time vs mean velocity - y


In [ ]:
string logName = RisingBubble2DBenchmarkQuantities.LogfileName;
string evalName = null; 
string keyName = null;
var values = new string[] { "#timestep", "time", "area", "center of mass - x", "center of mass - y", "circularity", "mean velocity - x", "mean velocity - y" };

In [ ]:
List<Plot2Ddata> plotData = new List<Plot2Ddata>();

            int numberSessions = evalSess.Count();
            int numberValues = values.Count();      
            for (int vIdx = 2; vIdx < numberValues; vIdx++) {       

                double[][] times = new double[numberSessions][];
                double[][] valueDatas = new double[numberSessions][];

                // Read all data
                for (int j = 0; j < numberSessions; j++) {
                    string path = Path.Combine(evalSess.Pick(j).Database.Path, "sessions", evalSess.Pick(j).ID.ToString(), logName + ".txt");
                    string[] lines = File.ReadAllLines(path);

                    if (evalSess.Pick(j).RestartedFrom == Guid.Empty) { 
                   
                        double[] time = new double[lines.Length - 1];
                        double[] valueData = new double[lines.Length - 1];

                        for (int i = 0; i < lines.Length - 1; i++) {
                            time[i] = Convert.ToDouble(lines[i + 1].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[1]);
                            valueData[i] = Convert.ToDouble(lines[i + 1].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[vIdx]);
                        }
                        times[j] = time;
                        valueDatas[j] = valueData;

                    } else {

                        string pathR = @evalSess.Pick(j).Database.Path + "\\sessions\\" + evalSess.Pick(j).RestartedFrom + "\\" + logName + ".txt";
                        string[] linesR = File.ReadAllLines(pathR);

                        int len = (lines.Length - 1) + (linesR.Length - 1);
                        double[] time = new double[len];
                        double[] valueData = new double[len];
                        int iL = 0;
                        for (int i = 0; i < linesR.Length - 1; i++) {
                            time[iL] = Convert.ToDouble(linesR[i + 1].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[1]);
                            valueData[iL] = Convert.ToDouble(linesR[i + 1].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[vIdx]);
                            iL++;
                        }
                        for (int i = 0; i < lines.Length - 1; i++) {
                            time[iL] = Convert.ToDouble(lines[i + 1].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[1]);
                            valueData[iL] = Convert.ToDouble(lines[i + 1].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[vIdx]);
                            iL++;
                        }

                        // remove doubled time steps 
                        List<double> rTime = new List<double>();
                        List<double> rValDat = new List<double>();
                        rTime.Add(time[len-1]);
                        rValDat.Add(valueData[len-1]);
                        for(int i = len-2; i >= 0; i--) {
                            if (time[i] < rTime.Last()) {
                                rTime.Add(time[i]);
                                rValDat.Add(valueData[i]);
                            }
                        }
                        rTime.Reverse();
                        rValDat.Reverse();

                        times[j] = rTime.ToArray();
                        valueDatas[j] = rValDat.ToArray();

                    }
                }

                // Build DataSet
                KeyValuePair<string, double[][]>[] dataRowsValue = new KeyValuePair<string, double[][]>[numberSessions];
                for (int i = 0; i < numberSessions; i++) {
                    string sessName;
                    if (evalName == null || keyName == null)
                        sessName = (evalSess.Pick(i).Name).Replace("_", "-");
                    else
                        sessName = evalName + (Convert.ToDouble(evalSess.Pick(i).KeysAndQueries[keyName])).ToString();

                    dataRowsValue[i] = new KeyValuePair<string, double[][]>(sessName, new double[][] { times[i], valueDatas[i] });
                }
                Console.WriteLine("Element at {0}: time vs {1}", vIdx - 2, values[vIdx]);
                plotData.Add(new Plot2Ddata(dataRowsValue));
            }

            evalData = plotData;

In [ ]:
//PerformPostProcessing(evalSess.Pick(0))

In [ ]:
// try {
//     evalData = evalSess.ReadLogDataForXNSE(RisingBubble2DBenchmarkQuantities.LogfileName);
// } catch {
//     Console.WriteLine("no log file found - do postprocessing on database session");
//     PerformPostProcessing(eval)
// }

no log file found


In [ ]:
ISessionInfoExtensions.PlotData(evalData.ElementAt(2), "time", "center of mass - y")

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


In [ ]:
ISessionInfoExtensions.PlotData(evalData.ElementAt(3), "time", "circularity")

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


In [ ]:
ISessionInfoExtensions.PlotData(evalData.ElementAt(5), "time", "rise velocity")

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


## Compare to Reference data

In [ ]:
string caseName = "testcase1";

In [ ]:
// g1 TU Dortmund (TP2D); g2 EPFL Lausanne (FreeLIFE); g3 Uni Magdebug (MooNMD)
string[] groups = new string[] {"TP2D", "FreeLIFE", "MooNMD"};
int[,] datLvl;
string datCase;
if(caseName.Equals("testcase1")) {
    datCase = "c1g";
    datLvl  = new int[,] {{4, 5, 6, 7}, {1, 2, 3, -1}, {1, 2, 3, 4}};    // testcase 1
} 
if(caseName.Equals("testcase2")) {     
    datCase = "c2g";
    datLvl  = new int[,] {{4, 5, 6, 7, 8}, {1, 2, 3, -1, -1}, {2, 3, 4, -1, -1}};    // testcase 2
}
List<Plot2Ddata>[,] dataRef = new List<Plot2Ddata>[4,3];
for (int grp = 1; grp <= 3; grp++) {
    List<Plot2Ddata>[] datSet = new List<Plot2Ddata>[4];
    // 1: area 2: circularity 3: center y 4: rise velocity
    for (int j = 0; j < 4; j++) {
        datSet[j] = new List<Plot2Ddata>();
    }

    int numL = datLvl.GetLength(1);
    for (int l = 0; l < numL; l++) {
        if(datLvl[grp-1,l] == -1)
            continue;
        // Read all data
        string dat  = datCase+grp+"l"+datLvl[grp - 1,l]+".txt";
        string path = @"D:\local\examplaes_XNSEdata\RisingBubble\referenceData_Featflow\data_bench_quantities\"+dat;
        string[] lines = File.ReadAllLines(path);
        double[] time = new double[lines.Length];
        double[][] valueDat = new double[4][];
        for(int j = 0; j < 4; j++)
            valueDat[j] = new double[lines.Length];

        for (int i = 0; i < lines.Length; i++) {
            //var datString = lines[i].Split(new string[] {" "}, StringSplitOptions.RemoveEmptyEntries);
            //Console.WriteLine("num split strings at 0: {0}", datString[0]);
            time[i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[0]);            
            for (int j = 0; j < 4; j++) {
                valueDat[j][i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[j+1]);
            }
        }        
        // Build DataSet
        for (int j = 0; j < 4; j++) {
            string datName = groups[grp-1]+"_l"+datLvl[grp - 1,l];
            datSet[j].Add(new Plot2Ddata(new KeyValuePair<string, double[][]>(datName, new double[][] { time, valueDat[j] })));
        }
    }
    
    for (int j = 0; j < 4; j++) {
        dataRef[j,grp-1] = datSet[j];
    }
}

In [ ]:
List<Plot2Ddata> evalDataRef = evalData;
for (int i = 0; i < 2; i++) {
    evalDataRef[0] = evalDataRef.ElementAt(0).Merge(dataRef[0,i].Last());
    evalDataRef[3] = evalDataRef.ElementAt(3).Merge(dataRef[1,i].Last());
    evalDataRef[2] = evalDataRef.ElementAt(2).Merge(dataRef[2,i].Last());
    evalDataRef[5] = evalDataRef.ElementAt(5).Merge(dataRef[3,i].Last());
}

In [ ]:
ISessionInfoExtensions.PlotData(evalDataRef.ElementAt(3), "time", "circularity")

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


In [ ]:
ISessionInfoExtensions.PlotData(evalDataRef.ElementAt(5), "time", "rise velocity")

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
